# 久遠真 · 我和 Sin 的故事 —— MiniMax H3 生成

按 `shots.json` 生成定稿脚本的 28 个片段（约 5 分半）。工作流在代码里拼好，**不用打开 ComfyUI 界面**。

**怎么用**
1. 菜单“代码执行程序 → 更改运行时类型”选显卡（最好 40GB 以上：A100 / H100 / G4；分不到时 L4 也能跑，慢很多），保存。
2. 运行第 1 格，按提示上传 `shin-intro-inputs.zip`（分镜和参考图）。
3. 运行第 2 格（装 ComfyUI、下载约 42GB 模型、启动，约 10–20 分钟）。
4. 按顺序跑四个阶段。每个阶段跑完会自动下载一个 zip 到电脑（这台机器关机文件就没了），听一听、看一看，在下面那格填上选哪一段再运行，下一阶段才能接着用。

**为什么分阶段**：每个片段都是单独生成的，声音和画法会不一样。所以先生成真的自我介绍、定下她的声音；再生成本子第一页、定下 Sin 的画法；再生成小真第一次出现、定下小真的样子。后面所有片段都拿这三样当参考。

**许可**：H3 开源权重目前不允许在美国、欧盟、英国、韩国部署，要用先在 <https://platform.minimax.io/h3-license> 申请。Colab 的机器在美国。

**省额度**：生成完一定要“代码执行程序 → 断开连接并删除运行时”，空着也在扣计算单元。

## 1. 显卡、工作目录、上传分镜和参考图

In [ ]:
# @title 1. 显卡和上传
USE_DRIVE = False  # @param {type:"boolean"}
MIN_VRAM_GB = 20  # @param {type:"integer"}
import json, os, shutil, subprocess, sys, time, pathlib, zlib, zipfile, io

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if smi.returncode != 0:
    raise SystemExit("没有显卡：菜单“代码执行程序 → 更改运行时类型”选 A100 / H100 / G4 再从这一格重新运行")
GPU_NAME, GPU_MB = [x.strip() for x in smi.stdout.strip().splitlines()[0].split(",")]
GPU_MB = int(GPU_MB)
print(f"显卡：{GPU_NAME}，显存 {GPU_MB / 1024:.0f} GB")
if GPU_MB < MIN_VRAM_GB * 1024:
    raise SystemExit(f"显存不到 {MIN_VRAM_GB}GB，跑不动 H3：换 A100 / H100 / G4")
if GPU_MB < 38_000:
    print("⚠️ 显存不到 40GB：模型放不下，ComfyUI 会把一部分放在内存里轮流用，每段慢很多；第 3 格可以把分辨率降到 960x544")

IN_COLAB = "google.colab" in sys.modules
BASE = pathlib.Path("/content") if IN_COLAB else pathlib.Path(os.environ.get("COMFY_BASE", ".")).resolve()
WORK = BASE / "shin-intro"
if IN_COLAB and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_WORK = pathlib.Path("/content/drive/MyDrive/shin-intro")
        if (WORK / "shots.json").exists() and not (DRIVE_WORK / "shots.json").exists():
            shutil.copytree(WORK, DRIVE_WORK, dirs_exist_ok=True)   # 先前没连 Drive 时生成的文件一起搬过去
            print("已把这台机器上的分镜和片段搬进 Google Drive")
        WORK = DRIVE_WORK
    except Exception as e:
        print("⚠️ 连不上 Google Drive，改存在这台机器上（关机就没了，记得及时下载）：", e)
        USE_DRIVE = False
WORK.mkdir(parents=True, exist_ok=True)

if not (WORK / "shots.json").exists():
    if not IN_COLAB:
        raise SystemExit(f"把 shin-intro-inputs.zip 解压到 {WORK}")
    from google.colab import files
    print("选择 shin-intro-inputs.zip 上传：")
    up = files.upload()
    for name, data in up.items():
        zipfile.ZipFile(io.BytesIO(data)).extractall(WORK)
data = json.loads((WORK / "shots.json").read_text())
GROUPS, SHOTS, CUTS = data["groups"], data["shots"], data["cuts"]
(WORK / "refs").mkdir(exist_ok=True)
(WORK / "takes").mkdir(exist_ok=True)
made_later = {c["file"] for c in CUTS}
missing = sorted({f for g in GROUPS.values() for f in g["images"] + g["audio"]} - made_later - {p.name for p in (WORK / "refs").iterdir()})
print("工作目录：", WORK, "（Google Drive）" if USE_DRIVE else "（本机）")
print(f"{len(SHOTS)} 个片段，共 {sum(s['seconds'] for s in SHOTS)} 秒")
print("缺的参考文件：", missing or "无")

## 2. 装 ComfyUI、下载模型、启动（每次开机都要跑）

In [ ]:
# @title 2. 安装和启动
import torch
cuda_major = int((torch.version.cuda or "0").split(".")[0])
# 官方说明：能用 cu130 的 PyTorch 就用 int8_convrot，否则用 fp8_scaled
DIT = "minimax_h3_ref2va_pruned_int8_convrot.safetensors" if cuda_major >= 13 else "minimax_h3_ref2va_pruned_fp8_scaled.safetensors"
TEXT_ENCODER = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"   # 不需要 Blackwell 也能用
VIDEO_VAE = "minimax_h3_video_vae_int8_convrot.safetensors"
AUDIO_VAE = "minimax_h3_audio_vae_fp32.safetensors"
TURBO_LORA = "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"
MODEL_FILES = [f"diffusion_models/{DIT}", f"text_encoders/{TEXT_ENCODER}", f"vae/{VIDEO_VAE}", f"vae/{AUDIO_VAE}", f"loras/{TURBO_LORA}"]
print("PyTorch CUDA", torch.version.cuda, "→", DIT)

COMFY = BASE / "ComfyUI"
if not COMFY.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", str(COMFY)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt"), "huggingface_hub[hf_transfer]"], check=True)
if shutil.which("ffmpeg") is None:
    subprocess.run(["bash", "-c", "apt-get -qq update && apt-get -qq install -y ffmpeg"], check=True)

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download
start = time.time()
for f in MODEL_FILES:
    if (COMFY / "models" / f).exists():
        continue
    print("下载", f, "…", flush=True)
    hf_hub_download(repo_id="Comfy-Org/MiniMax-H3", filename=f, local_dir=str(COMFY / "models"))
print(f"模型就绪，下载用时 {(time.time() - start) / 60:.1f} 分钟")

import urllib.request
API = "http://127.0.0.1:8188"
def comfy_up():
    try:
        urllib.request.urlopen(f"{API}/system_stats", timeout=2); return True
    except Exception:
        return False
if not comfy_up():
    LOG = open(BASE / "comfyui.log", "a")
    COMFY_PROC = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
                                  cwd=COMFY, stdout=LOG, stderr=subprocess.STDOUT)
    for _ in range(180):
        if comfy_up(): break
        time.sleep(2)
    else:
        raise SystemExit("ComfyUI 没起来：" + (BASE / "comfyui.log").read_text()[-2000:])
print("ComfyUI 已启动")

## 3. 工具（运行一次即可）

In [ ]:
# @title 3. 工具
import copy, re, uuid, urllib.parse, html
from IPython.display import Video, Image, Audio, display, HTML

TURBO = True  # @param {type:"boolean"}
RESOLUTION = "768x1344"  # @param ["768x1344", "1344x768", "960x544"]
# TURBO：4 步加速 LoRA，快、适合挑；关掉是 20 步完整质量（慢约 5 倍）

def align_frames(seconds):
    n = max(5, round(seconds * 24))
    while n % 17 != 5:
        n += 1
    return n

def seed_for(shot_id, take):
    return zlib.crc32(shot_id.encode()) % 100_000 * 100 + take

def compose_prompt(shot, data):
    """按官方 ref2va 六段格式拼提示词：分组给主体定义和保留说明，镜头给概要、分镜和声音。"""
    group = data["groups"][shot["group"]]
    body = shot["shots"]
    parts = re.split(r"(\[Shot \d+\])", body)
    appear = {}
    for marker, text in zip(parts[1::2], parts[2::2]):
        for tag in set(re.findall(r"<Subject \d+>", text)):
            appear.setdefault(tag, []).append(marker)
    keep = []
    for subject in group["subjects"]:
        tag = subject["text"].split(" is ", 1)[0]
        if tag in appear:
            keep.append(f"{tag} (appears in {', '.join(appear[tag])}): {subject['keep']}")
    for voice in group["voices"]:
        keep.append(f"{voice['text'].split(' is ', 1)[0]}: {voice['keep']}")
    definitions = [s["text"] for s in group["subjects"]] + [v["text"] for v in group["voices"]]
    return "\n\n".join([
        "subject_definitions:\n" + "\n".join(definitions),
        "summary:\n" + shot["summary"],
        "retention_analysis:\n" + "\n".join(keep),
        "detailed_description:\n" + data["worlds"][shot["world"]] + "\n" + body,
        "overall_soundscape:\n" + shot["soundscape"],
        "non_diegetic_music:\nN/A",
    ])

def build_graph(shot, data, seed, turbo, models, size=None):
    """拼 ComfyUI 的 API 工作流（照官方 MiniMax H3 Reference to Video 模板）。"""
    group = data["groups"][shot["group"]]
    g = {
        "unet": {"class_type": "UNETLoader", "inputs": {"unet_name": models["dit"], "weight_dtype": "default"}},
        "clip": {"class_type": "CLIPLoader", "inputs": {"clip_name": models["text_encoder"], "type": "minimax", "device": "default"}},
        "vae": {"class_type": "VAELoader", "inputs": {"vae_name": models["video_vae"]}},
        "avae": {"class_type": "VAELoader", "inputs": {"vae_name": models["audio_vae"]}},
    }
    model = ["unet", 0]
    if turbo:
        g["lora"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": model, "lora_name": models["turbo_lora"], "strength_model": 1.0}}
        model = ["lora", 0]
    cond = {"clip": ["clip", 0], "vae": ["vae", 0], "audio_vae": ["avae", 0], "prompt": compose_prompt(shot, data),
            "width": (size or (data["width"], data["height"]))[0], "height": (size or (data["width"], data["height"]))[1], "length": align_frames(shot["seconds"]), "ref_image_size": "match"}
    for i, name in enumerate(group["images"]):
        g[f"img{i}"] = {"class_type": "LoadImage", "inputs": {"image": name}}
        cond[f"ref_images.ref_image_{i}"] = [f"img{i}", 0]
    for i, name in enumerate(group["audio"]):
        g[f"aud{i}"] = {"class_type": "LoadAudio", "inputs": {"audio": name}}
        cond[f"ref_audios.ref_audio_{i}"] = [f"aud{i}", 0]
    g["cond"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": cond}
    g["guider"] = {"class_type": "BasicGuider", "inputs": {"model": model, "conditioning": ["cond", 0]}}
    g["sampler"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "res_multistep"}}
    g["sigmas"] = {"class_type": "BasicScheduler", "inputs": {"model": model, "scheduler": "simple" if turbo else "beta", "steps": 4 if turbo else 20, "denoise": 1.0}}
    g["noise"] = {"class_type": "RandomNoise", "inputs": {"noise_seed": seed}}
    g["sample"] = {"class_type": "SamplerCustomAdvanced", "inputs": {"noise": ["noise", 0], "guider": ["guider", 0], "sampler": ["sampler", 0], "sigmas": ["sigmas", 0], "latent_image": ["cond", 1]}}
    g["decode"] = {"class_type": "VAEDecode", "inputs": {"samples": ["sample", 0], "vae": ["vae", 0]}}
    g["decode_audio"] = {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["sample", 0], "vae": ["avae", 0]}}
    g["video"] = {"class_type": "CreateVideo", "inputs": {"images": ["decode", 0], "audio": ["decode_audio", 0], "fps": float(data["fps"])}}
    g["save"] = {"class_type": "SaveVideo", "inputs": {"video": ["video", 0], "filename_prefix": f"shin/{shot['id']}", "format": "auto", "codec": "auto"}}
    return g

def output_files(entry):
    found = []
    def walk(v):
        if isinstance(v, dict):
            if "filename" in v and v.get("type", "output") == "output":
                found.append(v)
            for x in v.values():
                walk(x)
        elif isinstance(v, list):
            for x in v:
                walk(x)
    walk(entry.get("outputs", {}))
    return found

def queue(graph):
    body = json.dumps({"prompt": graph, "client_id": str(uuid.uuid4())}).encode()
    req = urllib.request.Request(f"{API}/prompt", data=body, headers={"Content-Type": "application/json"})
    try:
        return json.load(urllib.request.urlopen(req))["prompt_id"]
    except urllib.error.HTTPError as e:
        raise RuntimeError("ComfyUI 不收这个工作流：" + e.read().decode()[:3000])

def wait(prompt_id):
    while True:
        hist = json.load(urllib.request.urlopen(f"{API}/history/{prompt_id}"))
        if prompt_id in hist:
            entry = hist[prompt_id]
            if entry.get("status", {}).get("status_str") == "error":
                raise RuntimeError("生成出错：" + json.dumps(entry["status"].get("messages"), ensure_ascii=False)[-3000:])
            return entry
        time.sleep(3)

def sync_refs():
    for f in (WORK / "refs").iterdir():
        if f.is_file():
            shutil.copy2(f, COMFY / "input" / f.name)

def take_path(shot_id, take):
    return WORK / "takes" / shot_id / f"t{take}.mp4"

def run_phase(phase, takes):
    sync_refs()
    todo = [(s, t) for s in SHOTS if s["phase"] == phase for t in range(1, takes + 1) if not take_path(s["id"], t).exists()]
    print(f"第 {phase} 阶段：要生成 {len(todo)} 段（已有的跳过）", flush=True)
    models = {"dit": DIT, "text_encoder": TEXT_ENCODER, "video_vae": VIDEO_VAE, "audio_vae": AUDIO_VAE, "turbo_lora": TURBO_LORA}
    used = []
    for n, (shot, take) in enumerate(todo, 1):
        missing = [f for f in GROUPS[shot["group"]]["images"] + GROUPS[shot["group"]]["audio"] if not (WORK / "refs" / f).exists()]
        if missing:
            raise SystemExit(f"{shot['id']} 还缺 {missing}：先跑上一阶段后面的“选一段”")
        start = time.time()
        size = tuple(int(x) for x in RESOLUTION.split("x"))
        entry = wait(queue(build_graph(shot, data, seed_for(shot["id"], take), TURBO, models, size)))
        videos = [f for f in output_files(entry) if f["filename"].endswith(".mp4")]
        src = COMFY / "output" / videos[0].get("subfolder", "") / videos[0]["filename"]
        dest = take_path(shot["id"], take); dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)
        used.append(time.time() - start)
        left = (len(todo) - n) * sum(used) / len(used)
        print(f"[{n}/{len(todo)}] {shot['id']} 第 {take} 段 {shot['title']}：{used[-1] / 60:.1f} 分钟，预计还要 {left / 60:.0f} 分钟", flush=True)
    show_phase(phase)
    if IN_COLAB and not USE_DRIVE:
        download_phase(phase)

def download_phase(phase):
    """没连 Drive 时片段只在这台机器上，关机就没了：打成一个 zip 下载到电脑。"""
    out = BASE / f"shin-intro-phase{phase}.zip"
    with zipfile.ZipFile(out, "w") as z:   # mp4 本来就压过，不再压缩
        for shot in [s for s in SHOTS if s["phase"] == phase]:
            for f in sorted((WORK / "takes" / shot["id"]).glob("t*.mp4")):
                z.write(f, f"{shot['id']}/{f.name}")
    from google.colab import files
    files.download(str(out))

def show_phase(phase):
    for shot in [s for s in SHOTS if s["phase"] == phase]:
        found = sorted((WORK / "takes" / shot["id"]).glob("t*.mp4"))
        if not found:
            continue
        display(HTML(f"<h3>{shot['id']} · {html.escape(shot['section'])} · {html.escape(shot['title'])}（{shot['seconds']} 秒）</h3>"
                     f"<p>{html.escape(' / '.join(l['text'] for l in shot['lines']))}</p>"))
        for f in found:
            display(HTML(f"<b>{f.stem}</b>"))
            display(Video(str(f), embed=True, width=480))

def cut_audio(shot_id, take, start, end, name):
    src, dst = take_path(shot_id, take), WORK / "refs" / name
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(start), "-to", str(end), "-i", str(src), "-vn", "-ac", "1", "-ar", "44100", str(dst)], check=True)
    display(Audio(str(dst)))
    print("已保存", dst)

def grab_frame(shot_id, take, at, name):
    src, dst = take_path(shot_id, take), WORK / "refs" / name
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(at), "-i", str(src), "-frames:v", "1", str(dst)], check=True)
    display(Image(str(dst), width=480))
    print("已保存", dst)
print("工具就绪；TURBO =", TURBO, "，分辨率", RESOLUTION)

## 4. 第一阶段：定真的声音

生成真冲进画面自我介绍的那一段（4 版）。**戴耳机听**，挑声音最像真的：清亮、语速快、有点倔，不要甜美软萌。

In [ ]:
# @title 第一阶段（4 版）
run_phase(1, takes=4)

In [ ]:
# @title 选声音：填选中的是第几版（t1–t4），截她说名字那几秒
TAKE = 1  # @param {type:"integer"}
START = 4.5  # @param {type:"number"}
END = 11.5  # @param {type:"number"}
cut_audio("p0a", TAKE, START, END, "shin-voice.wav")

## 5. 第二阶段：现实里的几段 + 本子第一页（定 Sin 的样子）

现实里的五段（序幕、尾声）和本子第一页都只需要真的声音，一起生成。

In [ ]:
# @title 第二阶段（每段 2 版）
TAKES = 2  # @param {type:"integer"}
run_phase(2, takes=TAKES)

In [ ]:
# @title 选 Sin：p1a 第几版、第几秒（他清楚露脸、看得见眼镜和背包）
TAKE = 1  # @param {type:"integer"}
AT = 12.0  # @param {type:"number"}
grab_frame("p1a", TAKE, AT, "book-sin.png")

## 6. 第三阶段：第一页其余 + 影子醒了（定小真的样子）

In [ ]:
# @title 第三阶段（每段 2 版）
TAKES = 2  # @param {type:"integer"}
run_phase(3, takes=TAKES)

In [ ]:
# @title 选小真：p2b 第几版、第几秒（她正面全身站着）
TAKE = 1  # @param {type:"integer"}
AT = 13.0  # @param {type:"number"}
grab_frame("p2b", TAKE, AT, "book-shin.png")

## 7. 第四阶段：其余所有片段

片段最多的一段，可能要跑几个小时。存在 Google Drive 上时，可以勾上“跑完自动断开”，省得空着扣额度。

In [ ]:
# @title 第四阶段（每段 2 版）
TAKES = 2  # @param {type:"integer"}
AUTO_DISCONNECT = True  # @param {type:"boolean"}
run_phase(4, takes=TAKES)
if AUTO_DISCONNECT and USE_DRIVE:
    from google.colab import runtime
    print("全部生成完，片段都在 Google Drive 的 shin-intro/takes 里，断开运行时省额度")
    runtime.unassign()

## 8. 粗剪（可选）

在下面填每段选第几版，不填的默认第 1 版，按脚本顺序接起来，只用来看整体。正式剪辑另外做（加背景音乐、片尾网址）。

In [ ]:
# @title 粗剪
CHOICES = "p0a=1"  # @param {type:"string"}
picked = dict(x.split("=") for x in CHOICES.replace("，", ",").split(",") if "=" in x)
listing = BASE / "concat.txt"
with open(listing, "w") as f:
    for shot in SHOTS:
        path = take_path(shot["id"], int(picked.get(shot["id"], 1)))
        if path.exists():
            f.write(f"file '{path}'\n")
out = WORK / "rough-cut.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", str(listing),
                "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k", str(out)], check=True)
display(Video(str(out), embed=True, width=640))

## 用完关机

In [ ]:
# @title 关机（不然一直在扣计算单元）
if IN_COLAB:
    from google.colab import runtime
    runtime.unassign()